In [ ]:
# Upload files and dataset

In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/val.csv")
test_df = pd.read_csv("/content/test.csv")

In [ ]:
# Variation #1: Using TF-IDF Preprocessing and MLP Model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# TF-IDF Preprocessing
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_df["query"]).toarray()
X_val = vectorizer.transform(val_df["query"]).toarray()
X_test = vectorizer.transform(test_df["query"]).toarray()

y_train = train_df["carb"].values
y_val = val_df["carb"].values

# Dataset class
class NutriDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# Dataloaders
train_loader = DataLoader(NutriDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(NutriDataset(X_val, y_val), batch_size=64)
test_loader = DataLoader(NutriDataset(X_test), batch_size=64)

# MLP model
class MLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super(MLPRegressor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLPRegressor(input_dim=X_train.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop
for epoch in range(20):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

    # Evaluate
    model.eval()
    with torch.no_grad():
        val_preds_1 = []
        for X_batch, _ in val_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).cpu().numpy()
            val_preds_1.append(preds)
        val_preds_1 = np.concatenate(val_preds_1)
        rmse = np.sqrt(mean_squared_error(y_val, val_preds_1))
        print(f"Epoch {epoch+1}: Validation RMSE = {rmse:.4f}")


In [ ]:
# Variation #2: Using Sentence Transformer Preprocessing and MLP Model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Sentence Transformers Preprocessing
model_st = SentenceTransformer("all-MiniLM-L6-v2")

X_train = model_st.encode(train_df["query"].tolist(), show_progress_bar=True)
X_val = model_st.encode(val_df["query"].tolist(), show_progress_bar=True)
X_test = model_st.encode(test_df["query"].tolist(), show_progress_bar=True)

y_train = train_df["carb"].values
y_val = val_df["carb"].values

# Dataset Class
class NutriDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (self.X[idx], self.y[idx]) if self.y is not None else self.X[idx]

# Dataloaders
train_loader = DataLoader(NutriDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(NutriDataset(X_val, y_val), batch_size=32, shuffle=False)
test_loader = DataLoader(NutriDataset(X_test), batch_size=32, shuffle=False)

# MLP Model
class MLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super(MLPRegressor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(1)

# Training Loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLPRegressor(input_dim=X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(20):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_preds_2 = []
        for X_batch, _ in val_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).cpu().numpy()
            val_preds_2.append(preds)
        val_preds_2 = np.concatenate(val_preds_2)
        rmse = np.sqrt(mean_squared_error(y_val, val_preds_2))
        print(f"Epoch {epoch+1}: Validation RMSE = {rmse:.4f}")


In [ ]:
# Validation #3: Using Sentence Transformer Preprocessing and RNN Model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Sentence Transformers Preprocessing
model_st = SentenceTransformer("all-MiniLM-L6-v2")

X_train = model_st.encode(train_df["query"].tolist(), show_progress_bar=True)
X_val = model_st.encode(val_df["query"].tolist(), show_progress_bar=True)
X_test = model_st.encode(test_df["query"].tolist(), show_progress_bar=True)

y_train = train_df["carb"].values
y_val = val_df["carb"].values

# Dataset Class
class NutriEmbedDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # Shape: [batch, seq_len=1, emb_dim]
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return self.X.size(0)

    def __getitem__(self, idx):
        return (self.X[idx], self.y[idx]) if self.y is not None else self.X[idx]

# Dataloader
train_loader = DataLoader(NutriEmbedDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(NutriEmbedDataset(X_val, y_val), batch_size=32)
test_loader = DataLoader(NutriEmbedDataset(X_test), batch_size=32)

# RNN Model
class RNNRegressor(nn.Module):
    def __init__(self, input_dim, hidden_size=64, num_layers=1):
        super(RNNRegressor, self).__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.fc(hidden[-1]).squeeze(1)

# Training Loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RNNRegressor(input_dim=X_train.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(20):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_preds_3 = []
    with torch.no_grad():
        for X_batch, _ in val_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).cpu().numpy()
            val_preds_3.append(preds)
    val_preds_3 = np.concatenate(val_preds_3)
    rmse = np.sqrt(mean_squared_error(y_val, val_preds_3))
    print(f"Epoch {epoch+1}: Validation RMSE = {rmse:.4f}")


In [ ]:
# Evalute which method is best performing

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np


# Compute RMSEs
results = {
    "TF-IDF + MLP": np.sqrt(mean_squared_error(y_val, val_preds_1)),
    "ST + MLP": np.sqrt(mean_squared_error(y_val, val_preds_2)),
    "ST + RNN": np.sqrt(mean_squared_error(y_val, val_preds_3)),
}

# Display results
print("Validation RMSEs:")
for name, rmse in results.items():
    print(f"{name}: {rmse:.4f}")

best = min(results, key=results.get)
print(f"\n✅ Best-performing approach: {best}")


In [ ]:
# Prediction using Variation #1 (TF-IDF + MLP)
model.eval()
with torch.no_grad():
    test_preds = []
    for X_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).cpu().numpy()
        test_preds.append(preds)
    test_preds = np.concatenate(test_preds)

# Save predictions
test_df["carbs"] = test_preds
test_df.to_csv("test_with_predictions_mlp.csv", index=False)
print("Predictions saved to test_with_predictions_mlp.csv")